## Problem 1: Comparative Edge Detection on Natural Images

Download any outdoor natural image (trees, buildings, roads).  
Apply Sobel, Laplacian, and Canny edge detection.  
Compare the outputs and write a short analysis explaining which operator captures the most meaningful edges and why.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load an outdoor image
img = cv2.imread("building.jpg")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Smooth small noise before edge detection
blur = cv2.GaussianBlur(gray, (5, 5), 1)

# Sobel edges in x and y directions
sobel_x = cv2.Sobel(blur, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(blur, cv2.CV_64F, 0, 1, ksize=3)
sobel = cv2.addWeighted(cv2.convertScaleAbs(sobel_x), 0.5,
                        cv2.convertScaleAbs(sobel_y), 0.5, 0)

# Laplacian and Canny edges
laplacian = cv2.convertScaleAbs(cv2.Laplacian(blur, cv2.CV_64F, ksize=3))
canny = cv2.Canny(blur, 80, 160)

# Compare the outputs
plt.figure(figsize=(12, 8))
for i, (image, title, cmap) in enumerate([
    (img_rgb, "Original Image", None),
    (sobel, "Sobel", "gray"),
    (laplacian, "Laplacian", "gray"),
    (canny, "Canny", "gray")
], 1):
    plt.subplot(2, 2, i)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")
plt.tight_layout()
plt.show()

# Print a brief comparison based on detected edge pixels
print("Edge pixel counts — Sobel:", np.count_nonzero(sobel),
      "| Laplacian:", np.count_nonzero(laplacian),
      "| Canny:", np.count_nonzero(canny))
print("Observation: Canny usually gives thin, connected edges and suppresses some noise; "
      "Sobel shows gradient strength, while Laplacian can highlight fine detail and noise.")
print("Conclusion: For clearly separated meaningful outlines, Canny is often the easiest to interpret; "
      "the best result depends on the image and threshold settings.")

## Problem 2: Noise Sensitivity Experiment

Take a clean grayscale image and intentionally add Gaussian noise and salt-and-pepper noise.  
Apply Sobel and Laplacian on both noisy versions and evaluate:

- Which operator is more affected by noise?
- How does edge clarity change?
- What preprocessing step improves the results?

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load a clean grayscale image
gray = cv2.imread("building.jpg", cv2.IMREAD_GRAYSCALE)
rng = np.random.default_rng(42)

# Add Gaussian noise
gaussian_noise = rng.normal(0, 20, gray.shape)
gaussian_img = np.clip(gray.astype(np.float32) + gaussian_noise, 0, 255).astype(np.uint8)

# Add salt-and-pepper noise
sp_img = gray.copy()
random_map = rng.random(gray.shape)
sp_img[random_map < 0.02] = 0
sp_img[random_map > 0.98] = 255

# Function to compare Sobel and Laplacian
def edge_maps(image):
    sx = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    sy = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    sobel = cv2.convertScaleAbs(cv2.addWeighted(sx, 0.5, sy, 0.5, 0))
    lap = cv2.convertScaleAbs(cv2.Laplacian(image, cv2.CV_64F, ksize=3))
    return sobel, lap

sobel_g, lap_g = edge_maps(gaussian_img)
sobel_sp, lap_sp = edge_maps(sp_img)

# Apply Gaussian blur as preprocessing
gaussian_blur = cv2.GaussianBlur(gaussian_img, (5, 5), 1)
sp_blur = cv2.medianBlur(sp_img, 3)
sobel_g_blur, lap_g_blur = edge_maps(gaussian_blur)
sobel_sp_blur, lap_sp_blur = edge_maps(sp_blur)

# Display noisy images and edge maps
plt.figure(figsize=(13, 10))
items = [
    (gaussian_img, "Gaussian Noise"), (sobel_g, "Sobel: Gaussian"),
    (lap_g, "Laplacian: Gaussian"), (sp_img, "Salt-and-Pepper Noise"),
    (sobel_sp, "Sobel: Salt-and-Pepper"), (lap_sp, "Laplacian: Salt-and-Pepper"),
    (sobel_g_blur, "Sobel after Gaussian Blur"), (lap_g_blur, "Laplacian after Gaussian Blur"),
    (sobel_sp_blur, "Sobel after Median Blur"), (lap_sp_blur, "Laplacian after Median Blur")
]
for i, (image, title) in enumerate(items, 1):
    plt.subplot(4, 3, i)
    plt.imshow(image, cmap="gray")
    plt.title(title)
    plt.axis("off")
plt.tight_layout()
plt.show()

print("Observation: Laplacian is generally more sensitive to noise because it uses a second derivative.")
print("Edge clarity: Noise creates extra bright responses and makes real edges harder to distinguish.")
print("Preprocessing: Gaussian blur helps with Gaussian noise; median blur is effective for salt-and-pepper noise.")
print("Sobel usually responds less sharply to noise than Laplacian, but both improve after suitable denoising.")

## Problem 3: Compression Ratio Comparison (RLE vs Huffman)

Load a grayscale image with large smooth regions (e.g., document scan or sky image).  
Compress it using Run-Length Encoding and Huffman Coding separately.  
Calculate and compare:

- Bit-length before compression
- Bit-length after compression
- Percentage reduction

Explain why each technique performed the way it did.

In [ ]:
import cv2
import numpy as np
from collections import Counter
import heapq

# Load grayscale image and flatten its pixels
img = cv2.imread("pattern.jpg", cv2.IMREAD_GRAYSCALE)
pixels = img.flatten()
original_bits = len(pixels) * 8

# Run-Length Encoding stores (pixel value, run length)
rle = []
count = 1
for i in range(1, len(pixels)):
    if pixels[i] == pixels[i - 1]:
        count += 1
    else:
        rle.append((int(pixels[i - 1]), count))
        count = 1
rle.append((int(pixels[-1]), count))
# Estimate RLE storage: 8 bits for value + 32 bits for run length per pair
rle_bits = len(rle) * (8 + 32)

# Build Huffman codes from pixel frequencies
frequency = Counter(int(p) for p in pixels)
heap = [[freq, [value, ""]] for value, freq in frequency.items()]
heapq.heapify(heap)

if len(heap) == 1:
    huffman_codes = {heap[0][1][0]: "0"}
else:
    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        for pair in left[1:]:
            pair[1] = "0" + pair[1]
        for pair in right[1:]:
            pair[1] = "1" + pair[1]
        heapq.heappush(heap, [left[0] + right[0]] + left[1:] + right[1:])
    huffman_codes = dict(heap[0][1:])

huffman_bits = sum(frequency[value] * len(code) for value, code in huffman_codes.items())

# Calculate percentage reduction
rle_reduction = (1 - rle_bits / original_bits) * 100
huffman_reduction = (1 - huffman_bits / original_bits) * 100

print("Original bit-length:", original_bits)
print("RLE compressed bit-length (estimated):", rle_bits)
print("RLE percentage reduction:", round(rle_reduction, 2), "%")
print("Huffman encoded bit-length (code bits only):", huffman_bits)
print("Huffman percentage reduction (code bits only):", round(huffman_reduction, 2), "%")
print("Observation: RLE works best when neighbouring pixels repeat in long runs; varied textures create many short runs.")
print("Observation: Huffman coding uses shorter codes for frequent pixel values, so its result depends on the image histogram.")
print("Note: These estimates exclude headers, dictionaries/code tables, and file-format overhead, so they are not full file sizes.")

## Problem 4: JPEG Block Artifact Analysis

Implement an 8×8 block-based JPEG-style compression (DCT + quantisation).  
Compress the same image using low, medium, and high quantisation levels.  
Analyse:

- Where block artifacts appear
- Why some textures lose detail
- How PSNR and SSIM change at each level

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

# Load image and convert it to RGB and YCrCb
img_bgr = cv2.imread("city.jpg")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
ycc = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
Y, Cr, Cb = cv2.split(ycc)
Y = np.float32(Y)

# JPEG luminance quantisation matrix
Q = np.array([
    [16,11,10,16,24,40,51,61],
    [12,12,14,19,26,58,60,55],
    [14,13,16,24,40,57,69,56],
    [14,17,22,29,51,87,80,62],
    [18,22,37,56,68,109,103,77],
    [24,35,55,64,81,104,113,92],
    [49,64,78,87,103,121,120,101],
    [72,92,95,98,112,100,103,99]
], dtype=np.float32)

# Pad image so every block is 8 by 8
h, w = Y.shape
pad_h = (8 - h % 8) % 8
pad_w = (8 - w % 8) % 8
Y_pad = cv2.copyMakeBorder(Y, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)
levels = [("Low", 0.5), ("Medium", 1.0), ("High", 2.0)]
results = []

for name, scale in levels:
    q_matrix = np.maximum(Q * scale, 1)
    compressed = np.zeros_like(Y_pad)

    # Apply DCT and quantisation to each 8 by 8 block
    for i in range(0, Y_pad.shape[0], 8):
        for j in range(0, Y_pad.shape[1], 8):
            block = Y_pad[i:i+8, j:j+8] - 128
            dct_block = cv2.dct(block)
            quantized = np.round(dct_block / q_matrix)
            restored = cv2.idct(quantized * q_matrix) + 128
            compressed[i:i+8, j:j+8] = restored

    compressed = np.clip(compressed[:h, :w], 0, 255).astype(np.uint8)
    restored_ycc = cv2.merge([compressed, Cr, Cb])
    restored_bgr = cv2.cvtColor(restored_ycc, cv2.COLOR_YCrCb2BGR)
    restored_rgb = cv2.cvtColor(restored_bgr, cv2.COLOR_BGR2RGB)

    mse = np.mean((img_rgb.astype(np.float32) - restored_rgb.astype(np.float32)) ** 2)
    psnr = float("inf") if mse == 0 else 10 * np.log10((255 ** 2) / mse)
    ssim_value = ssim(img_rgb, restored_rgb, channel_axis=2, data_range=255)
    results.append((name, restored_rgb, mse, psnr, ssim_value))

# Display original and the three reconstructed versions
plt.figure(figsize=(14, 8))
plt.subplot(2, 2, 1)
plt.imshow(img_rgb); plt.title("Original"); plt.axis("off")
for i, (name, restored, mse, psnr, ssim_value) in enumerate(results, 2):
    plt.subplot(2, 2, i)
    plt.imshow(restored)
    plt.title(f"{name} Quantisation\nPSNR={psnr:.2f} dB, SSIM={ssim_value:.4f}")
    plt.axis("off")
plt.tight_layout()
plt.show()

# Print quality metrics and observations
for name, restored, mse, psnr, ssim_value in results:
    print(f"{name} quantisation — MSE: {mse:.2f}, PSNR: {psnr:.2f} dB, SSIM: {ssim_value:.4f}")
print("Observation: Increasing quantisation strength removes more DCT detail and usually lowers PSNR and SSIM.")
print("Block artifacts may appear near 8×8 block boundaries; fine textures and sharp transitions are often affected first.")

## Problem 5: Texture-Based Segmentation using GLCM Features

Load a synthetic or natural image that contains multiple textures (wood, fabric, leaves, tiles).  
Extract GLCM features (contrast, energy, homogeneity) for each pixel neighbourhood.  
Segment the image into k = 3 clusters using these features.  
Write observations on how feature choice influences segmentation quality.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import graycomatrix, graycoprops
from sklearn.cluster import KMeans
from scipy.ndimage import generic_filter

# Load image and resize to keep local GLCM calculation manageable
img_bgr = cv2.imread("fruits.jpg")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
gray = cv2.resize(gray, (128, 128))
img_rgb = cv2.resize(img_rgb, (128, 128))
gray_levels = (gray // 16).astype(np.uint8)  # Reduce to 16 gray levels

# Calculate local GLCM features in each 5 by 5 neighbourhood
def local_glcm_feature(values, feature_name):
    patch = values.reshape(5, 5).astype(np.uint8)
    glcm = graycomatrix(patch, distances=[1], angles=[0], levels=16,
                        symmetric=True, normed=True)
    return graycoprops(glcm, feature_name)[0, 0]

contrast = generic_filter(gray_levels, lambda p: local_glcm_feature(p, "contrast"),
                          size=5, mode="nearest")
energy = generic_filter(gray_levels, lambda p: local_glcm_feature(p, "energy"),
                        size=5, mode="nearest")
homogeneity = generic_filter(gray_levels, lambda p: local_glcm_feature(p, "homogeneity"),
                             size=5, mode="nearest")

# Scale features and cluster pixels into three groups
features = np.stack([contrast, energy, homogeneity], axis=2)
features = (features - features.mean(axis=(0, 1))) / (features.std(axis=(0, 1)) + 1e-8)
pixels = features.reshape((-1, 3))
labels = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(pixels)
segmented = labels.reshape(gray.shape)

# Display texture feature maps and segmentation
plt.figure(figsize=(12, 8))
for i, (image, title, cmap) in enumerate([
    (img_rgb, "Original Image", None),
    (contrast, "GLCM Contrast", "gray"),
    (energy, "GLCM Energy", "gray"),
    (homogeneity, "GLCM Homogeneity", "gray"),
    (segmented, "K-Means Segmentation (k=3)", "viridis")
], 1):
    plt.subplot(2, 3, i)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")
plt.tight_layout()
plt.show()

print("Feature ranges — Contrast:", round(float(contrast.min()), 3), "to", round(float(contrast.max()), 3))
print("Feature ranges — Energy:", round(float(energy.min()), 3), "to", round(float(energy.max()), 3))
print("Feature ranges — Homogeneity:", round(float(homogeneity.min()), 3), "to", round(float(homogeneity.max()), 3))
print("Observation: Contrast helps separate rough texture, energy reflects repeated patterns, and homogeneity is higher in more uniform neighbourhoods.")
print("Observation: Combining the three features can separate textures better than relying on only one feature; similar textures may still be grouped together.")

## Problem 6: Multi-Feature Segmentation for Disease Detection

Load a plant or skin lesion image.  
Construct a feature vector combining:

- Intensity
- Gradient magnitude (Sobel)
- Local texture variance

Segment the image using K-means or Mean Shift.  
Evaluate:

- Which feature contributed most to separating diseased vs healthy regions
- Any misclassifications and why they occurred

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Load plant disease image
img_bgr = cv2.imread("leaf_disease.jpg")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# Calculate Sobel gradient magnitude
sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)
sobel_mag = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX)

# Calculate local 3 by 3 texture variance
gray_float = gray.astype(np.float32)
mean = cv2.blur(gray_float, (3, 3))
mean_sq = cv2.blur(gray_float ** 2, (3, 3))
local_var = np.maximum(mean_sq - mean ** 2, 0)
local_var = cv2.normalize(local_var, None, 0, 255, cv2.NORM_MINMAX)

# Combine the three features and cluster pixels
feature_vector = np.stack([gray, sobel_mag, local_var], axis=2).astype(np.float32)
h, w = gray.shape
pixels = feature_vector.reshape((-1, 3))
pixels = (pixels - pixels.mean(axis=0)) / (pixels.std(axis=0) + 1e-8)
labels = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(pixels)
segmented = labels.reshape((h, w))

# Display original image, features, and segmentation
plt.figure(figsize=(13, 8))
for i, (image, title, cmap) in enumerate([
    (img_rgb, "Original Leaf", None),
    (gray, "Intensity", "gray"),
    (sobel_mag, "Sobel Gradient Magnitude", "gray"),
    (local_var, "Local Texture Variance", "gray"),
    (segmented, "K-Means Segmentation (3 Features)", "nipy_spectral")
], 1):
    plt.subplot(2, 3, i)
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis("off")
plt.tight_layout()
plt.show()

# Print observations from feature variation
print("Feature standard deviations — Intensity:", round(float(np.std(gray)), 2),
      "| Sobel:", round(float(np.std(sobel_mag)), 2),
      "| Local variance:", round(float(np.std(local_var)), 2))
print("Observation: Intensity separates regions with different brightness; Sobel highlights boundaries and spots; local variance highlights uneven or textured areas.")
print("Likely useful feature: Local variance and Sobel magnitude can help highlight irregular lesions, while intensity helps when diseased and healthy regions differ in colour/brightness.")
print("Possible misclassifications: Shadows, leaf veins, reflections, and healthy rough areas can resemble disease. K-means clusters features without disease labels, so clusters must be interpreted visually.")